<a href="https://colab.research.google.com/github/div652/GPT-2/blob/main/gpt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Goal here is to set up GPT2 architecture
- reproduce results
- ability to load their model
- ability to train our own model
- We will reproduce the 124M model.
- smallest was 117M in their paper, was corrected to 124M later. see github.
- Will refer GPT3 as well, since there is not a huge departure in the architecture.
- Will implement in pytorch, but the initial code was in tensorflow.
- Their a torch implementation in huggingface, and you can load the model from here as well.
- https://huggingface.co/openai-community/gpt2


# Sources
Attention is all you need paper : https://arxiv.org/abs/1706.03762
OpenAI GPT-2 Paper : https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf
GPT2 Blogpost : https://openai.com/index/better-language-models/

OpenAI GPT-3 Paper :https://arxiv.org/pdf/2005.14165

We begin by setting up the model

# Imports

In [9]:
import os
import math
import time
import inspect
from dataclasses import dataclass # for GPT Config class
import torch
import torch.nn as nn
from torch.nn import functional as F

# Importing Trained Model

In [10]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
HuggingFaceModel = GPT2LMHeadModel.from_pretrained("gpt2")
HuggingFaceStateDict = HuggingFaceModel.state_dict()
for k,v in HuggingFaceStateDict.items():
  print(k,v.shape)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


transformer.wte.weight torch.Size([50257, 768])
transformer.wpe.weight torch.Size([1024, 768])
transformer.h.0.ln_1.weight torch.Size([768])
transformer.h.0.ln_1.bias torch.Size([768])
transformer.h.0.attn.c_attn.weight torch.Size([768, 2304])
transformer.h.0.attn.c_attn.bias torch.Size([2304])
transformer.h.0.attn.c_proj.weight torch.Size([768, 768])
transformer.h.0.attn.c_proj.bias torch.Size([768])
transformer.h.0.ln_2.weight torch.Size([768])
transformer.h.0.ln_2.bias torch.Size([768])
transformer.h.0.mlp.c_fc.weight torch.Size([768, 3072])
transformer.h.0.mlp.c_fc.bias torch.Size([3072])
transformer.h.0.mlp.c_proj.weight torch.Size([3072, 768])
transformer.h.0.mlp.c_proj.bias torch.Size([768])
transformer.h.1.ln_1.weight torch.Size([768])
transformer.h.1.ln_1.bias torch.Size([768])
transformer.h.1.attn.c_attn.weight torch.Size([768, 2304])
transformer.h.1.attn.c_attn.bias torch.Size([2304])
transformer.h.1.attn.c_proj.weight torch.Size([768, 768])
transformer.h.1.attn.c_proj.bias 

Below is a trial run to generate from the model

In [11]:
#sample from the model
from transformers import pipeline, set_seed
generator = pipeline('text-generation', model='gpt2')
set_seed(42)
generator("Hello, I'm a language model,", max_new_tokens=30, num_return_sequences=5)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "Hello, I'm a language model, I'm a language model. In my mind, I'm doing the same thing as you. All these different people are thinking about the same thing."},
 {'generated_text': 'Hello, I\'m a language model, and I\'m trying to understand what you\'re saying. In English, I have two English words: "l" and "es." In Dutch,'},
 {'generated_text': "Hello, I'm a language model, so I don't get much of a chance to interact with objects. But I did find that I can do things like check the state of a specific"},
 {'generated_text': 'Hello, I\'m a language model, a language model, a language model."'},
 {'generated_text': "Hello, I'm a language model, not a toolkit.\n\nIn a nutshell, I need to give language models a set of properties that I could use to describe them in real"}]

# Implementing our own model

## Hyper Parameter Setup

In [12]:
@dataclass
class GPTConfig:
  block_size: int=1024 # context length
  vocab_size: int=50257 # 50,000 BPE merges, 256 bytes, 1 EOS
  n_layer: int =12
  n_head: int =12
  n_embd: int =768

## Model Code

In [13]:
class CausalSelfAttention(nn.Module):
# c_attn.weight torch.Size([768, 2304])
# c_attn.bias torch.Size([2304])
# c_proj.weight torch.Size([768, 768])
# c_proj.bias torch.Size([768])
  def __init__(self,config):
    super().__init__()
    assert config.n_embd%config.n_head==0
    self.n_embd = config.n_embd
    self.n_head = config.n_head

  # using bias here because openAI used it, I don't think it is needed.
    self.c_attn = nn.Linear(self.n_embd, 3*self.n_embd)
    self.c_prof = nn.Linear(self.n_embd, self.n_embd)

    # calling it mask,
    self.register_buffer("mask", torch.tril(torch.ones(config.block_size,config.block_size)).view(1,1,config.block_size,config.block_size))

  def forward(self,x):
    B,T,C = x.shape
    qkv = self.c_attn(x)
    q,k,v = torch.split(qkv, self.n_embd, dim=2)

    q = q.view(B,T,self.n_head, C//self.n_head).transpose(1,2) # B,n_h,T,n_embd/n_h
    k = k.view(B,T,self.n_head, C//self.n_head).transpose(1,2) # B,n_h,T,n_embd/n
    v = v.view(B,T,self.n_head, C//self.n_head).transpose(1,2) # B,n_h,T,n_embd/n_h

    att = q@k.transpose(-2,-1) # B,n_h, T, T
    att = att*((C//self.n_head)**-0.5) # scaling by head dimesion
    att = att.masked_fill(self.mask[:,:,T,T]==0,float('-inf'))
    att = F.softmax(att,dim=-1)
    y = att@v
    y = y.transpose(1,2).contiguous().view(B,T,C)
    y = self.c_proj(y)
    return y




nn.Gelu has two versions, the original version, and he approximate version calculated from tanh.

https://arxiv.org/pdf/1606.08415was deveopled the erf function which is used in torch for GELU implementation , was very slow in torch at that time. So at that time bert and gpt-2 used the tanh version in their architecutre, so we are going to use that.

Why GELU vs. RELU
- activations that ended up in the zero region had no gradient
- so there is always some contribution in the GELU case, there are not dead regions


In [14]:
class FeedForwardLayer(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.c_fc = nn.Linear(config.n_embd, 4*config.n_embd)
    self.gelu = nn.GELU(approximate='tanh')
    self.c_proj = nn.Linear(4*config.n_embd, config.n_embd)

  def forward(self,x):
    x = self.c_fc(x)
    x = self.gelu(x)
    x = self.c_proj(x)
    return x


Note: in the original Attention is all you need paper there was this one difference.
- Layer Norms were after Attention block and MLP . They are now before attention and MLP


In [25]:
class Block(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.ln_1 = nn.LayerNorm(config.n_embd)
    self.attn = CausalSelfAttention(config)
    self.ln_2 = nn.LayerNorm(config.n_embd)
    self.mlp = FeedForwardLayer(config)

# each block is just a repeated application of map-reduce.
  def forward(self,x):
    # attention is the reduce
    x = x + self.attn(self.ln_1(x))
    # mlp is the map
    x = x + self.mlp(self.ln_2(x))
    return x

### Note on why we use no_grad while doing copy
1. Avoiding Computation Graph Overhead
By default, if sd[k] or sd_hf[k] are tensors that have requires_grad=True (which is standard for model parameters), any operation performed on them—including .copy_() or .t() (transpose)—could be tracked by the Autograd engine.

Without no_grad: PyTorch might try to build a "graph" of this operation, consuming extra memory and CPU/GPU cycles to keep track of how the data was moved.

With no_grad: You tell PyTorch, "This is just a memory-to-memory copy; don't worry about the history of these tensors."

2. Preventing "Leaf Tensor" Errors
In PyTorch, model parameters are usually leaf tensors. Autograd generally prevents you from performing "in-place" operations on leaf tensors that require gradients because it would break the ability to calculate gradients for them later.
If you tried to use .copy_() on a parameter without no_grad(), PyTorch might throw a runtime error saying:

"RuntimeError: a view of a leaf Variable that requires grad is being used in an in-place operation."

Using the no_grad context manager signals that you are intentionally modifying the values outside of the training loop.

In [27]:
class GPT(nn.Module):
  def __init__(self,config):
    super().__init__()

    self.transformer = nn.ModuleDict({
        "wte" : nn.Embedding(config.vocab_size, config.n_embd),
        "wpe" : nn.Embedding(config.block_size, config.n_embd),
        "h" : nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
        "ln_f" : nn.LayerNorm(config.n_embd)
    })
    self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
# from_pretrained is a constructor that returns a GPT object initialised with
# weights form the hugging face model, based on the model_type we provide it with.
  @classmethod
  def from_pretrained(cls,model_type):
    # Loads pretrained GPT-2 Model weight from hugging face
    assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
    from transformers import GPT2LMHeadModel
    print("Loading Weights from pretrained gpt: %s" % model_type)

    # n_layer, n_head and n_embd are detemrined from model type
    config_args = {
        'gpt2': dict(n_layer=12, n_head=12, n_embd=768),        # 124M params
        'gpt2-medium': dict(n_layer=24, n_head=16, n_embd=1024),# 350M params
        'gpt2-large': dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
        'gpt2-xl': dict(n_layer=48, n_head=25, n_embd=1600),    # 1558M params
    }[model_type]
    config_args['vocab_size'] = 50257 # same for all the models
    config_args['block_size'] = 1024  # same for all the models
    config = GPTConfig(**config_args)

    # our model
    model = GPT(config)
    sd = model.state_dict()
    sd_keys = sd.keys()
    print(sd_keys,sep='\n')
    sd_keys = [k for k in sd_keys if not k.endswith(".attn.mask")] # ignoring the register buffers, these are non trainable parameters

    # hugging face model
    model_hf = GPT2LMHeadModel.from_pretrained(model_type)
    sd_hf = model_hf.state_dict()
    sd_keys_hf = sd_hf.keys()
    print(sd_keys_hf,sep='\n')
    sd_keys_hf = [k for k in sd_keys_hf if not k.endswith(".attn.masked_bias")] # ignoring the register buffers, these are non trainable parameters
    sd_keys_hf = [k for k in sd_keys_hf if not k.endswith(".attn.bias")] # ignoring the register buffers, these are non trainable parameters
    transposed = [
        'c_attn.weight',
        'c_attn.bias',
        'c_proj.weight',
        'c_proj.bias',
        'wpe.weight'
    ]
    # some layers in the gpt2 implementation need to transposed since they were originally written as a conv1d.
    # while we have implemented it using a vanilla Linear layer
    assert(len(sd_keys)==len(sd_keys_hf),f"Model sizes do not match {len(sd_keys)} vs {len(sd_keys_hf)}")
    for k in sd_keys:
      if any(k.endswith(w) for w in transposed):
        assert(sd_hf[k].shape[::-1]==sd[k].shape)
        with torch.no_grad():
          sd[k].copy_(sd_hf[k].t())
      else:
        #vanilla copt over the parameters
        assert(sd_hf[k].shape==sd[k].shape)
        with torch.no_grad():
          sd[k].copy_(sd_hf[k])

    return model


In [29]:
gpt = GPT.from_pretrained('gpt2')
print("model loaded successfully")

Loading Weights from pretrained gpt: gpt2
odict_keys(['transformer.wte.weight', 'transformer.wpe.weight', 'transformer.h.0.ln_1.weight', 'transformer.h.0.ln_1.bias', 'transformer.h.0.attn.mask', 'transformer.h.0.attn.c_attn.weight', 'transformer.h.0.attn.c_attn.bias', 'transformer.h.0.attn.c_prof.weight', 'transformer.h.0.attn.c_prof.bias', 'transformer.h.0.ln_2.weight', 'transformer.h.0.ln_2.bias', 'transformer.h.0.mlp.c_fc.weight', 'transformer.h.0.mlp.c_fc.bias', 'transformer.h.0.mlp.c_proj.weight', 'transformer.h.0.mlp.c_proj.bias', 'transformer.h.1.ln_1.weight', 'transformer.h.1.ln_1.bias', 'transformer.h.1.attn.mask', 'transformer.h.1.attn.c_attn.weight', 'transformer.h.1.attn.c_attn.bias', 'transformer.h.1.attn.c_prof.weight', 'transformer.h.1.attn.c_prof.bias', 'transformer.h.1.ln_2.weight', 'transformer.h.1.ln_2.bias', 'transformer.h.1.mlp.c_fc.weight', 'transformer.h.1.mlp.c_fc.bias', 'transformer.h.1.mlp.c_proj.weight', 'transformer.h.1.mlp.c_proj.bias', 'transformer.h.2.ln_

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


odict_keys(['transformer.wte.weight', 'transformer.wpe.weight', 'transformer.h.0.ln_1.weight', 'transformer.h.0.ln_1.bias', 'transformer.h.0.attn.c_attn.weight', 'transformer.h.0.attn.c_attn.bias', 'transformer.h.0.attn.c_proj.weight', 'transformer.h.0.attn.c_proj.bias', 'transformer.h.0.ln_2.weight', 'transformer.h.0.ln_2.bias', 'transformer.h.0.mlp.c_fc.weight', 'transformer.h.0.mlp.c_fc.bias', 'transformer.h.0.mlp.c_proj.weight', 'transformer.h.0.mlp.c_proj.bias', 'transformer.h.1.ln_1.weight', 'transformer.h.1.ln_1.bias', 'transformer.h.1.attn.c_attn.weight', 'transformer.h.1.attn.c_attn.bias', 'transformer.h.1.attn.c_proj.weight', 'transformer.h.1.attn.c_proj.bias', 'transformer.h.1.ln_2.weight', 'transformer.h.1.ln_2.bias', 'transformer.h.1.mlp.c_fc.weight', 'transformer.h.1.mlp.c_fc.bias', 'transformer.h.1.mlp.c_proj.weight', 'transformer.h.1.mlp.c_proj.bias', 'transformer.h.2.ln_1.weight', 'transformer.h.2.ln_1.bias', 'transformer.h.2.attn.c_attn.weight', 'transformer.h.2.attn.